# One-Step Baseline Pipeline

This notebook provides a single end-to-end workflow for the first complete project milestone:

1. confirm the project environment and dependencies
2. verify the dataset structure
3. train the one-step baseline model
4. evaluate the trained model on the held-out test set
5. save report-ready outputs for later comparison with the two-step architecture

The notebook is designed for team use. It aims to be readable, reproducible, and aligned with the assignment requirements without becoming overly verbose.

## Project Context

Current dataset structure:

```text
prepared_dataset/
  images/
    train/
    test/
  labels/
    train/
    test/
  data.yaml
```

Project decisions used in this notebook:

- the provided labeled dataset is used for training
- the self-collected corrected dataset is used as the held-out test set
- we do not create an internal validation split for the assignment dataset
- for Ultralytics compatibility, a temporary runtime YAML uses `val: images/train`, but training is still run with `val=False`
- final benchmark numbers come from the held-out `test` split, not from training-time metrics

## Step 1 - Environment Setup

If the project environment has not been created yet, run these commands in the project root:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt
```

This keeps the project reproducible for every team member and avoids mixing packages from unrelated local experiments.

In [1]:
from pathlib import Path
import sys
import platform
import json

import yaml
import pandas as pd
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / 'prepared_dataset'
TRAIN_IMAGES = DATA_ROOT / 'images' / 'train'
TRAIN_LABELS = DATA_ROOT / 'labels' / 'train'
TEST_IMAGES = DATA_ROOT / 'images' / 'test'
TEST_LABELS = DATA_ROOT / 'labels' / 'test'
REQUIREMENTS_FILE = PROJECT_ROOT / 'requirements.txt'

RUNS_ROOT = PROJECT_ROOT / 'runs' / 'baseline_runs'
TRAIN_RUN_NAME = 'one_step_baseline'
TRAIN_RUN_DIR = RUNS_ROOT / TRAIN_RUN_NAME
WEIGHTS_PATH = TRAIN_RUN_DIR / 'weights' / 'best.pt'

EVAL_ROOT = PROJECT_ROOT / 'runs' / 'evaluation'
EVAL_RUN_NAME = 'one_step_test_eval'
EVAL_RUN_DIR = EVAL_ROOT / EVAL_RUN_NAME
PREDICTION_RUN_NAME = 'one_step_test_predictions'

TRAIN_RUNTIME_YAML = DATA_ROOT / 'train_runtime.data.yaml'
TEST_RUNTIME_YAML = DATA_ROOT / 'test_runtime.data.yaml'

CLASS_NAMES = {
    0: 'fall detected',
    1: 'walk',
    2: 'sit',
}

print('Python executable :', sys.executable)
print('Python version    :', sys.version.split()[0])
print('Platform          :', platform.platform())
print('Project root      :', PROJECT_ROOT)
print('Requirements file :', REQUIREMENTS_FILE)
print('Data root         :', DATA_ROOT)


Python executable : c:\Users\shr\Documents\GitHub\intelligent-system\.venv\Scripts\python.exe
Python version    : 3.13.9
Platform          : Windows-11-10.0.26200-SP0
Project root      : c:\Users\shr\Documents\GitHub\intelligent-system
Requirements file : c:\Users\shr\Documents\GitHub\intelligent-system\requirements.txt
Data root         : c:\Users\shr\Documents\GitHub\intelligent-system\prepared_dataset


## Step 2 - Dependency Check

The project dependency list is stored in `requirements.txt`. Keeping it small and explicit makes the project easier to reproduce, review, and hand off to teammates.

In [2]:
requirements_text = REQUIREMENTS_FILE.read_text(encoding='utf-8')
print(requirements_text)


jupyterlab
ultralytics
torch
torchvision
torchaudio
opencv-python
pandas
matplotlib
seaborn
pyyaml
scikit-learn
tqdm



## Step 3 - Dataset Verification

Before training or evaluation, confirm that the train and test splits are present and internally consistent.

In [3]:
# Count files in each split and ensure the number of images matches the number of label files.
counts = {
    'train_images': len(list(TRAIN_IMAGES.glob('*'))),
    'train_labels': len(list(TRAIN_LABELS.glob('*.txt'))),
    'test_images': len(list(TEST_IMAGES.glob('*'))),
    'test_labels': len(list(TEST_LABELS.glob('*.txt'))),
}

for key, value in counts.items():
    print(f'{key:>12}: {value}')

assert counts['train_images'] == counts['train_labels'], 'Train image/label count mismatch'
assert counts['test_images'] == counts['test_labels'], 'Test image/label count mismatch'


train_images: 485
train_labels: 485
 test_images: 274
 test_labels: 274


## Step 4 - Build Runtime YAML Files

Ultralytics expects a dataset YAML. We generate two runtime YAML files:

- a training runtime YAML
- a test runtime YAML

The training runtime YAML includes a temporary `val` key pointing to the training split because the loader expects it, even though the assignment setup does not use a real validation split.

In [4]:
train_yaml = {
    'path': str(DATA_ROOT),
    'train': 'images/train',
    'val': 'images/train',
    'test': 'images/test',
    'names': CLASS_NAMES,
}

test_yaml = {
    'path': str(DATA_ROOT),
    'train': 'images/train',
    'val': 'images/train',
    'test': 'images/test',
    'names': CLASS_NAMES,
}

TRAIN_RUNTIME_YAML.write_text(yaml.safe_dump(train_yaml, sort_keys=False), encoding='utf-8')
TEST_RUNTIME_YAML.write_text(yaml.safe_dump(test_yaml, sort_keys=False), encoding='utf-8')

print('Training runtime YAML:')
print(TRAIN_RUNTIME_YAML.read_text(encoding='utf-8'))
print('Test runtime YAML:')
print(TEST_RUNTIME_YAML.read_text(encoding='utf-8'))


Training runtime YAML:
path: c:\Users\shr\Documents\GitHub\intelligent-system\prepared_dataset
train: images/train
val: images/train
test: images/test
names:
  0: fall detected
  1: walk
  2: sit

Test runtime YAML:
path: c:\Users\shr\Documents\GitHub\intelligent-system\prepared_dataset
train: images/train
val: images/train
test: images/test
names:
  0: fall detected
  1: walk
  2: sit



## Step 5 - Train the One-Step Baseline

The one-step baseline uses a YOLO detector trained directly on the three action classes:

- `fall detected`
- `walk`
- `sit`

Recommended baseline settings:

- model: `yolov8n.pt`
- image size: `640`
- epochs: `100`
- batch size: `16`

If training has already been completed, this cell can be skipped.

In [5]:
MODEL_NAME = 'yolov8n.pt'

# Instantiate the baseline detector from pretrained YOLO weights.
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(TRAIN_RUNTIME_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    project=str(RUNS_ROOT),
    name=TRAIN_RUN_NAME,
    val=False,
)


Ultralytics 8.4.51  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\shr\Documents\GitHub\intelligent-system\prepared_dataset\train_runtime.data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=one_step_baseline-3

## Step 6 - Confirm Training Outputs

The most important training artifact is `weights/best.pt`, which will be used for held-out test evaluation and later demos.

In [6]:
if TRAIN_RUN_DIR.exists():
    print('Training output directory:')
    for path in sorted(TRAIN_RUN_DIR.iterdir()):
        print('-', path.name)
else:
    print('Training has not been run yet.')

print('Best weights path:', WEIGHTS_PATH)
print('Best weights exists:', WEIGHTS_PATH.exists())


Training output directory:
- args.yaml
- labels.jpg
- results.csv
- train_batch0.jpg
- train_batch1.jpg
- train_batch2.jpg
- weights
Best weights path: c:\Users\shr\Documents\GitHub\intelligent-system\runs\baseline_runs\one_step_baseline\weights\best.pt
Best weights exists: True


## Step 7 - Held-Out Test Evaluation

This is the first valid benchmark for the one-step architecture because the test set was kept separate from training.

These metrics should be used as the one-step baseline in the report and later compared against the two-step architecture.

In [7]:
eval_model = YOLO(str(WEIGHTS_PATH))

# Evaluate only on the held-out test split and save plots for later reporting.
test_metrics = eval_model.val(
    data=str(TEST_RUNTIME_YAML),
    split='test',
    project=str(EVAL_ROOT),
    name=EVAL_RUN_NAME,
    plots=True,
    save_json=True,
)


Ultralytics 8.4.51  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 673.192.8 MB/s, size: 2212.6 KB)
val: Scanning C:\Users\shr\Documents\GitHub\intelligent-system\prepared_dataset\labels\test... 274 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 274/274 909.4it/s 0.3s0.2s
val: New cache created: C:\Users\shr\Documents\GitHub\intelligent-system\prepared_dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.6it/s 5.0s0.2s
                   all        274        391      0.597      0.645       0.62      0.309
         fall detected         73         95      0.349      0.716       0.43      0.242
                  walk        128        177      0.759      0.723      0.816      0.454
                   sit         89        119      

## Step 8 - Summarize the Main Metrics

The key summary metrics are:

- precision
- recall
- mAP50
- mAP50-95

These are the core numbers to report for the one-step baseline.

In [8]:
summary = {
    'precision': float(test_metrics.box.mp),
    'recall': float(test_metrics.box.mr),
    'mAP50': float(test_metrics.box.map50),
    'mAP50-95': float(test_metrics.box.map),
}

for metric_name, metric_value in summary.items():
    print(f'{metric_name:>10}: {metric_value:.5f}')


 precision: 0.59727
    recall: 0.64492
     mAP50: 0.61970
  mAP50-95: 0.30877


## Step 9 - Per-Class Performance

Per-class results help identify whether the model behaves differently on `fall detected`, `walk`, and `sit`.

In [9]:
per_class_df = pd.DataFrame({
    'class_id': list(CLASS_NAMES.keys()),
    'class_name': list(CLASS_NAMES.values()),
    'precision': test_metrics.box.p.tolist(),
    'recall': test_metrics.box.r.tolist(),
    'mAP50': test_metrics.box.ap50.tolist(),
    'mAP50-95': test_metrics.box.ap.tolist(),
})

per_class_df


,class_id,class_name,precision,recall,mAP50,mAP50-95
0,0,fall detected,0.349407,0.715789,0.430188,0.241977
1,1,walk,0.759028,0.723164,0.815692,0.454261
2,2,sit,0.683371,0.495798,0.613211,0.230063


## Step 10 - Save Compact Evaluation Artifacts

These summary files are useful for later report writing and model comparison.

In [10]:
summary_path = EVAL_RUN_DIR / 'metrics_summary.json'
per_class_path = EVAL_RUN_DIR / 'per_class_metrics.csv'

EVAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
per_class_df.to_csv(per_class_path, index=False)

print('Saved summary to :', summary_path)
print('Saved per-class  :', per_class_path)


Saved summary to : c:\Users\shr\Documents\GitHub\intelligent-system\runs\evaluation\one_step_test_eval\metrics_summary.json
Saved per-class  : c:\Users\shr\Documents\GitHub\intelligent-system\runs\evaluation\one_step_test_eval\per_class_metrics.csv


## Step 11 - Save Prediction Visualizations

Numerical metrics should be supported by qualitative examples. This step saves prediction images that can later be used for discussion, failure analysis, and the final report.

In [11]:
# Use stream=True so predictions are processed incrementally instead of storing
# all result objects in notebook memory at once. This is much safer for VS Code
# notebook sessions on medium-sized image sets.
for _ in eval_model.predict(
    source=str(TEST_IMAGES),
    project=str(EVAL_ROOT),
    name=PREDICTION_RUN_NAME,
    save=True,
    save_txt=False,
    conf=0.25,
    imgsz=640,
    stream=True,
    verbose=False,
):
    pass

print('Saved prediction images to:', EVAL_ROOT / PREDICTION_RUN_NAME)


Results saved to C:\Users\shr\Documents\GitHub\intelligent-system\runs\evaluation\one_step_test_predictions
Saved prediction images to: c:\Users\shr\Documents\GitHub\intelligent-system\runs\evaluation\one_step_test_predictions


## What to Review After Running This Notebook

The team should inspect:

- `runs/baseline_runs/one_step_baseline/weights/best.pt`
- `runs/evaluation/one_step_test_eval/metrics_summary.json`
- `runs/evaluation/one_step_test_eval/per_class_metrics.csv`
- confusion matrices and plots saved by Ultralytics
- prediction images in `runs/evaluation/one_step_test_predictions/`

Once these results are reviewed, the next major milestone is implementing and evaluating the two-step architecture on the same test set.